In [4]:
from google.colab import files

uploaded = files.upload()

Saving Online Retail.xlsx to Online Retail (1).xlsx


In [5]:
import os

print(os.listdir("/content"))

['.config', 'Online Retail (1).xlsx', 'Online Retail.xlsx', 'sample_data']


In [6]:
import pandas as pd

online_retail = pd.read_excel("Online Retail.xlsx")

print(online_retail.shape)
online_retail.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
# Make a copy for KPI analysis
kpi_df = online_retail.copy()

# Convert InvoiceDate to datetime
kpi_df["InvoiceDate"] = pd.to_datetime(kpi_df["InvoiceDate"])

# Remove cancelled invoices
kpi_df = kpi_df[
    ~kpi_df["InvoiceNo"].astype(str).str.startswith("C")
]

# Keep only positive quantities and prices
kpi_df = kpi_df[
    (kpi_df["Quantity"] > 0) &
    (kpi_df["UnitPrice"] > 0)
]

print("Original rows:", len(online_retail))
print("Rows after cleaning:", len(kpi_df))

Original rows: 541909
Rows after cleaning: 530104


In [8]:
kpi_df["Revenue"] = kpi_df["Quantity"] * kpi_df["UnitPrice"]

total_revenue = kpi_df["Revenue"].sum()

print("Total Revenue:", total_revenue)

Total Revenue: 10666684.544


In [9]:
total_orders = kpi_df["InvoiceNo"].nunique()

print("Total Orders:", total_orders)

Total Orders: 19960


In [10]:
total_quantity = kpi_df["Quantity"].sum()

print("Total Quantity Sold:", total_quantity)

Total Quantity Sold: 5588376


In [11]:
total_products = kpi_df["StockCode"].nunique()

print("Total Products:", total_products)

Total Products: 3922


In [12]:
average_order_value = total_revenue / total_orders

print("Average Order Value:", average_order_value)

Average Order Value: 534.403033266533


In [13]:
top_products = (
    kpi_df.groupby("StockCode")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_products)

StockCode
23843     80995
23166     78033
22197     56921
84077     55047
85099B    48474
85123A    37660
84879     36461
21212     36419
23084     30788
22492     26633
Name: Quantity, dtype: int64


In [14]:
top_product_codes = (
    kpi_df.groupby("StockCode")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products = kpi_df[
    kpi_df["StockCode"].isin(top_product_codes.index)
][["StockCode", "Description"]].drop_duplicates()

top_products = top_products.merge(
    top_product_codes.rename("Quantity_Sold"),
    on="StockCode"
).sort_values("Quantity_Sold", ascending=False)

top_products

,StockCode,Description,Quantity_Sold
11,23843,"PAPER CRAFT , LITTLE BIRDIE",80995
7,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033
5,22197,SMALL POPCORN HOLDER,56921
9,22197,POPCORN HOLDER,56921
6,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,55047
4,85099B,JUMBO BAG RED RETROSPOT,48474
10,85123A,CREAM HANGING HEART T-LIGHT HOLDER,37660
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37660
1,84879,ASSORTED COLOUR BIRD ORNAMENT,36461
3,21212,PACK OF 72 RETROSPOT CAKE CASES,36419


In [15]:
top_revenue_products = (
    kpi_df.groupby(["StockCode", "Description"])["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_revenue_products)

StockCode  Description                       
DOT        DOTCOM POSTAGE                        206248.77
22423      REGENCY CAKESTAND 3 TIER              174484.74
23843      PAPER CRAFT , LITTLE BIRDIE           168469.60
85123A     WHITE HANGING HEART T-LIGHT HOLDER    104340.29
47566      PARTY BUNTING                          99504.33
85099B     JUMBO BAG RED RETROSPOT                94340.05
23166      MEDIUM CERAMIC TOP STORAGE JAR         81700.92
M          Manual                                 78110.27
POST       POSTAGE                                78101.88
23084      RABBIT NIGHT LIGHT                     66964.99
Name: Revenue, dtype: float64


In [16]:
# Create Year-Month column
kpi_df["YearMonth"] = kpi_df["InvoiceDate"].dt.to_period("M")

# Calculate monthly revenue
monthly_revenue = (
    kpi_df.groupby("YearMonth")["Revenue"]
    .sum()
    .reset_index()
)

print(monthly_revenue)

   YearMonth      Revenue
0    2010-12   823746.140
1    2011-01   691364.560
2    2011-02   523631.890
3    2011-03   717639.360
4    2011-04   537808.621
5    2011-05   770536.020
6    2011-06   761739.900
7    2011-07   719221.191
8    2011-08   759138.380
9    2011-09  1058590.172
10   2011-10  1154979.300
11   2011-11  1509496.330
12   2011-12   638792.680


In [17]:
monthly_revenue["Revenue_Growth_%"] = (
    monthly_revenue["Revenue"].pct_change() * 100
)

print(monthly_revenue)

   YearMonth      Revenue  Revenue_Growth_%
0    2010-12   823746.140               NaN
1    2011-01   691364.560        -16.070677
2    2011-02   523631.890        -24.261103
3    2011-03   717639.360         37.050354
4    2011-04   537808.621        -25.058650
5    2011-05   770536.020         43.273274
6    2011-06   761739.900         -1.141559
7    2011-07   719221.191         -5.581788
8    2011-08   759138.380          5.550057
9    2011-09  1058590.172         39.446272
10   2011-10  1154979.300          9.105424
11   2011-11  1509496.330         30.694665
12   2011-12   638792.680        -57.681733


In [18]:
country_revenue = (
    kpi_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(country_revenue)

Country
United Kingdom    9025222.084
Netherlands        285446.340
EIRE               283453.960
Germany            228867.140
France             209715.110
Australia          138521.310
Spain               61577.110
Switzerland         57089.900
Belgium             41196.340
Sweden              38378.330
Name: Revenue, dtype: float64


In [19]:
# Remove non-product/service entries
non_product_codes = ["POST", "DOT", "M"]

product_df = kpi_df[
    ~kpi_df["StockCode"].astype(str).isin(non_product_codes)
].copy()

print("Original KPI rows:", len(kpi_df))
print("Product rows:", len(product_df))

Original KPI rows: 530104
Product rows: 527951


In [20]:
product_df[["StockCode", "Description", "Quantity", "UnitPrice", "Revenue"]].head()

,StockCode,Description,Quantity,UnitPrice,Revenue
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,15.30
1,71053,WHITE METAL LANTERN,6,3.39,20.34
2,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,22.00
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,20.34
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,20.34


In [21]:
top_product_revenue = (
    product_df.groupby(["StockCode", "Description"])["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_product_revenue)

StockCode  Description                       
22423      REGENCY CAKESTAND 3 TIER              174484.74
23843      PAPER CRAFT , LITTLE BIRDIE           168469.60
85123A     WHITE HANGING HEART T-LIGHT HOLDER    104340.29
47566      PARTY BUNTING                          99504.33
85099B     JUMBO BAG RED RETROSPOT                94340.05
23166      MEDIUM CERAMIC TOP STORAGE JAR         81700.92
23084      RABBIT NIGHT LIGHT                     66964.99
22086      PAPER CHAIN KIT 50'S CHRISTMAS         64952.29
84879      ASSORTED COLOUR BIRD ORNAMENT          59094.93
79321      CHILLI LIGHTS                          54117.76
Name: Revenue, dtype: float64


In [22]:
top_product_quantity = (
    product_df.groupby(["StockCode", "Description"])["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_product_quantity)

StockCode  Description                       
23843      PAPER CRAFT , LITTLE BIRDIE           80995
23166      MEDIUM CERAMIC TOP STORAGE JAR        78033
84077      WORLD WAR 2 GLIDERS ASSTD DESIGNS     55047
85099B     JUMBO BAG RED RETROSPOT               48474
85123A     WHITE HANGING HEART T-LIGHT HOLDER    37599
22197      POPCORN HOLDER                        36761
84879      ASSORTED COLOUR BIRD ORNAMENT         36461
21212      PACK OF 72 RETROSPOT CAKE CASES       36419
23084      RABBIT NIGHT LIGHT                    30788
22492      MINI PAINT SET VINTAGE                26633
Name: Quantity, dtype: int64


In [23]:
total_customers = product_df["CustomerID"].nunique()

print("Total Customers:", total_customers)

Total Customers: 4335


In [24]:
average_revenue_per_customer = total_revenue / total_customers

print("Average Revenue per Customer:", average_revenue_per_customer)

Average Revenue per Customer: 2460.5962039215688


In [25]:
top_customers = (
    product_df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_customers)

CustomerID
14646.0    279138.02
18102.0    259657.30
17450.0    194550.79
16446.0    168472.50
14911.0    140450.72
12415.0    124564.53
14156.0    117379.63
17511.0     91062.38
12346.0     77183.60
16029.0     72882.09
Name: Revenue, dtype: float64


In [26]:
average_quantity_per_order = total_quantity / total_orders

print("Average Quantity per Order:", average_quantity_per_order)

Average Quantity per Order: 279.97875751503005


In [27]:
monthly_orders = (
    product_df.groupby("YearMonth")["InvoiceNo"]
    .nunique()
    .reset_index(name="Orders")
)

monthly_quantity = (
    product_df.groupby("YearMonth")["Quantity"]
    .sum()
    .reset_index(name="Quantity_Sold")
)

monthly_sales = monthly_orders.merge(
    monthly_quantity,
    on="YearMonth"
)

print(monthly_sales)

   YearMonth  Orders  Quantity_Sold
0    2010-12    1553         358764
1    2011-01    1082         387432
2    2011-02    1093         283256
3    2011-03    1441         377126
4    2011-04    1237         308528
5    2011-05    1669         395405
6    2011-06    1525         388843
7    2011-07    1452         401369
8    2011-08    1341         421460
9    2011-09    1821         569970
10   2011-10    2008         621747
11   2011-11    2753         750107
12   2011-12     817         313289


In [28]:
monthly_sales = monthly_sales.merge(
    monthly_revenue,
    on="YearMonth"
)

monthly_sales["Average_Order_Value"] = (
    monthly_sales["Revenue"] / monthly_sales["Orders"]
)

print(monthly_sales)

   YearMonth  Orders  Quantity_Sold      Revenue  Revenue_Growth_%  \
0    2010-12    1553         358764   823746.140               NaN   
1    2011-01    1082         387432   691364.560        -16.070677   
2    2011-02    1093         283256   523631.890        -24.261103   
3    2011-03    1441         377126   717639.360         37.050354   
4    2011-04    1237         308528   537808.621        -25.058650   
5    2011-05    1669         395405   770536.020         43.273274   
6    2011-06    1525         388843   761739.900         -1.141559   
7    2011-07    1452         401369   719221.191         -5.581788   
8    2011-08    1341         421460   759138.380          5.550057   
9    2011-09    1821         569970  1058590.172         39.446272   
10   2011-10    2008         621747  1154979.300          9.105424   
11   2011-11    2753         750107  1509496.330         30.694665   
12   2011-12     817         313289   638792.680        -57.681733   

    Average_Order_V

In [29]:
# Save the main KPI tables

monthly_sales.to_csv("monthly_sales_kpis.csv", index=False)
top_product_revenue.to_csv("top_product_revenue.csv")
top_product_quantity.to_csv("top_product_quantity.csv")
top_customers.to_csv("top_customers.csv")
country_revenue.to_csv("country_revenue.csv")

print("KPI tables saved successfully.")

KPI tables saved successfully.
